# lorentz transform visualization
### Física Electrónica - UTN FRA
*Cuaderno interactivo adaptado para Google Colab.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)


In [1]:
import numpy as np
import panel as pn
import plotly.graph_objs as go

pn.extension('plotly')

In [2]:
def plot_spacetime_surface(x_grid, y_grid, t_grid):
    surface = go.Surface(x=x_grid, y=y_grid, z=t_grid)
    layout = go.Layout(
        title='Lorentz transform of a 2-D square',
        autosize=False,
        width=600,
        height=500,
        margin=dict(t=50, b=50, r=50, l=50),
        scene_camera=dict(eye=dict(x=1.5, y=1.5, z=1)),
        scene = dict(
            xaxis = dict(nticks=8, range=[-10,10],),
            yaxis = dict(nticks=8, range=[-10,10],),
            zaxis = dict(nticks=8, range=[-5e-8,5e-8],),
            aspectratio = dict(x=1, y=1, z=1)
        ),
    )
    fig = dict(data=[surface], layout=layout)
    plotly_pane = pn.pane.Plotly(fig)
    return plotly_pane

In [3]:
xx = np.linspace(-1, 1, 2)
yy = np.linspace(-1, 1, 2)
x, y = np.meshgrid(xx, yy)
t = np.zeros((2, 2))

In [4]:
#from lorentz_transform import lorentz_transform
import numpy

# Perform a Lorentz transformation on a set of events.
#
# positions        a set of spatial positions
#                  1-D or greater
#                  positions.shape[-1] is the number of spatial dimensions to
#                  use in the transformation
#
# times            a set of times corresponding with each position
#                  0-D or greater
#                  shape must equal positions.shape[:-1]
#
# velocity         the velocity to transform the events by
#                  1-D
#                  shape must equal positions.shape[-1]
#
# speed_of_light   the maximum speed any object can move
#                  0-D
def lorentz_transform(positions, times, velocity, speed_of_light=300_000_000):
    position_shape = positions.shape
    num_spatial_dims = position_shape[-1]
    time_shape = times.shape

    if len(position_shape) != len(time_shape) + 1:
        raise ValueError("positions must have 1 more dim than time")

    if position_shape[:-1] != time_shape:
        raise ValueError("positions.shape[:-1] must equal times.shape")

    num_events = numpy.prod(time_shape)

    position_squeezed = positions.reshape(num_events, num_spatial_dims)
    time_squeezed = times.reshape(num_events)

    position_transformed, time_transformed = lorentz_transform_inner(position_squeezed, time_squeezed, velocity, speed_of_light)

    return position_transformed.reshape(position_shape), time_transformed.reshape(time_shape)


def lorentz_transform_inner(position, time, velocity, speed_of_light=300_000_000):
    speed_of_light_squ = numpy.double(speed_of_light)**2

    position = numpy.array(position, dtype=numpy.double)
    time = numpy.array(time, dtype=numpy.double)
    velocity = numpy.array(velocity, dtype=numpy.double)

    speed = numpy.linalg.norm(velocity, axis=-1)
    if speed == 0:
        return position, time

    if speed >= speed_of_light:
        raise ValueError("magnitude of velocity, %f, must be less than the speed of light, %f" % (
            speed, speed_of_light
        ))

    velocity_direction = velocity / speed

    lorentz_factor = numpy.double(1.0) / (numpy.double(1.0) - (speed * speed) / speed_of_light_squ)**numpy.double(0.5)

    # Project the position along the velocity
    projected_position = numpy.sum(position * velocity_direction, axis=-1)

    projected_position_transformed = lorentz_factor * (projected_position - speed * time)
    time_transformed = lorentz_factor * (time - speed * projected_position / speed_of_light_squ)

    # TODO: I don't like having two different operations here, there must be a generalized way to do it
    if projected_position.ndim == 0:
        # TODO: Right now, the caller won't exercise this part of the
        #       code, but it probably should, in the 1-dimensional space
        transformed_position_delta = (projected_position_transformed - projected_position) * velocity_direction
    elif projected_position.ndim >= 1:
        transformed_position_delta = numpy.outer(projected_position_transformed - projected_position, velocity_direction)


def plot_transformed_spacetime_surface(velocity_magnitude=0.0, velocity_angle=0.0):
    velocity = velocity_magnitude * np.array([
        np.cos(velocity_angle),
        np.sin(velocity_angle)
    ])
    
    positions = np.stack([x, y], axis=-1)
    pos_transformed, t_transformed = lorentz_transform(
        positions,
        t,
        velocity,
        speed_of_light=300_000_000

    )
    x_transformed = pos_transformed[:,:,0].squeeze()
    y_transformed = pos_transformed[:,:,1].squeeze()
    
    return plot_spacetime_surface(x_transformed, y_transformed, t_transformed)

In [ ]:
import panel.widgets as pnw

#velocity_magnitude  = pnw.FloatSlider(name='velocity_magnitude', value=0, start=0, end=0.99, step=0.05)
velocity_magnitude  = pnw.FloatSlider(name='velocity_magnitude', value=0, start=0, end=298_000_000, step=100_000)
velocity_angle  = pnw.FloatSlider(name='velocity_angle', value=0, start=0, end=2*np.pi)


@pn.depends(velocity_magnitude, velocity_angle)
def reactive_plot_transformed_spacetime_surface(velocity_magnitude, velocity_angle):
    return plot_transformed_spacetime_surface(velocity_magnitude, velocity_angle)

widgets   = pn.Column(velocity_magnitude, velocity_angle)
transformed_plot = pn.Row(reactive_plot_transformed_spacetime_surface, widgets)
transformed_plot

Row
    [0] ParamFunction(function)
    [1] Column
        [0] FloatSlider(end=298000000, name='velocity_magnitude', step=100000)
        [1] FloatSlider(end=6.283185307179586, name='velocity_angle')